# Bankruptcy Prediction Walkthrough

Step-by-step notebook for bankruptcy prediction on the Polish companies dataset.

The canonical implementation lives in the [`bankruptcy/`](bankruptcy/) package. [`bankruptcy_classifier.py`](bankruptcy_classifier.py) is the runnable entry point. This notebook imports those functions phase by phase.

**Run from the repo root** so `data/4year.arff` and `data/5year.arff` resolve correctly.

## Setup

In [ ]:
import matplotlib.pyplot as plt

from bankruptcy_classifier import (
    DATA_DIR,
    FIGURES_DIR,
    load_temporal_datasets,
    explore_datasets,
    prepare_features,
    evaluate_rf_baseline,
    tune_random_forest,
    evaluate_rf_tuned,
    top_rf_configurations,
    evaluate_xgb_baseline,
    tune_xgboost,
    fit_xgb_tuned,
    xgb_oof_predictions,
    top_xgb_configurations,
    plot_roc_comparison,
    plot_confusion_matrix,
    save_figures,
)

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print("Data:", DATA_DIR.resolve())
print("Figures:", FIGURES_DIR.resolve())

## Phase 1: Data loading

Load the 4th-year (train) and 5th-year (test) ARFF files.

In [ ]:
X_train_raw, y_train, X_test_raw, y_test, feature_names, class_name = load_temporal_datasets()
explore_datasets(X_train_raw, y_train, X_test_raw, y_test, feature_names)

## Phase 2: Preprocessing

Median imputation and standardization are fit on the 4th-year training data only, then applied to the 5th-year test set.

In [ ]:
X_train, X_test, imputer, scaler = prepare_features(
    X_train_raw, y_train, X_test_raw, y_test
)

## Phase 3: Random Forest

Baseline RF with out-of-fold CV, then hyperparameter tuning. Set `quick=True` for a fast demo; the full grid has 625 configurations.

In [ ]:
rf_baseline = evaluate_rf_baseline(X_train, y_train, X_test, y_test)
print(f"Baseline RF OOF CV AUC: {rf_baseline['cv_auc']:.4f}")
print(f"Baseline RF test AUC: {rf_baseline['test_auc']:.4f}")

rf_tune = tune_random_forest(X_train, y_train, quick=False, verbose=False)
rf_tuned = evaluate_rf_tuned(rf_tune["model"], X_train, y_train, X_test, y_test)
print(f"Tuned RF test AUC: {rf_tuned['test_auc']:.4f}")

for row in top_rf_configurations(rf_tune["tuning_results"]):
    print(row)

## Phase 4: XGBoost

Baseline and tuned XGBoost with early stopping inside CV folds.

In [ ]:
xgb_baseline = evaluate_xgb_baseline(X_train, y_train, X_test, y_test, verbose=False)

xgb_tune = tune_xgboost(X_train, y_train, quick=False, verbose=False)
xgb_tuned = fit_xgb_tuned(
    X_train, y_train, X_test, y_test, xgb_tune["best_params"], verbose=False
)
xgb_oof = xgb_oof_predictions(X_train, y_train, xgb_tune["best_params"], verbose=False)

for row in top_xgb_configurations(xgb_tune["tuning_results"]):
    print(row)

## Phase 5: Figures and metrics

Save figures for the report and display ROC/confusion matrices.

In [ ]:
save_figures(
    y_test,
    rf_baseline["y_test_proba"],
    rf_tuned["y_test_proba"],
    xgb_tuned["y_test_proba"],
)

roc_fig, aucs = plot_roc_comparison(
    y_test,
    rf_baseline["y_test_proba"],
    rf_tuned["y_test_proba"],
    xgb_tuned["y_test_proba"],
)
plt.show()

rf_cm_fig, _ = plot_confusion_matrix(y_test, rf_tuned["y_test_proba"], "Tuned RF Confusion Matrix")
plt.show()

xgb_cm_fig, _ = plot_confusion_matrix(
    y_test, xgb_tuned["y_test_proba"], "Tuned XGBoost Confusion Matrix"
)
plt.show()

## One-command pipeline

To run everything from the command line:

```bash
python bankruptcy_classifier.py
```